# RapierNet China-subset ablation table (final-comparable)

Runs component ablations of RapierNet (DySample / SARF / CoordGate) and evaluates them **exactly like the final protocol**: TTA @ 672px, conf=0.05, on the 682-image China test set.

**Key design choice** - every variant is seeded from the *final* `rapiernet_hr2/weights/best.pt` (mAP50 0.862). Because SARF/CoordGate are channel-preserving drop-ins and DySample is a parameter-free replacement, removing one module keeps *every other tensor shape-compatible*, so the whole trained model transfers and **only the ablated block differs**. Every row (including `Full`) is fine-tuned for `EPOCHS` epochs with the identical recipe, so all comparisons share the same budget.

Budget: `EPOCHS` (default 6) fine-tune epochs per variant @ 672px with the stage-3 recipe; each run keeps the same seed (42), SGD lr0=1e-3, warmup 1, close_mosaic 5, cls 0.8. Resume-safe (`last.pt` check).

Estimated wall time on a T4: ~1.2 h per variant at 6 epochs -> ~6 h for the 5 trained variants.

## What to upload (as a Kaggle dataset, one zip `rapiernet_resume.zip`):

```
rapi5_hr2/
  args.yaml
  rapiernet_hr2/   (or rapiernet_hr2/weights/*.pt - see restore cell below)
rapi5_hr2/rapiernet_hr2/args.yaml  results.csv  weights/best.pt  weights/last.pt
rapiernet_rdd2022_final.py
make_paper_figures.py
```

Simplest: take the contents of the folder `rapiernet_hr2` you downloaded (best.pt, last.pt, args.yaml, results.csv) and upload them as
```
rapiernet_hr2/weights/best.pt   rapiernet_hr2/weights/last.pt
rapiernet_hr2/args.yaml          rapiernet_hr2/results.csv
```
**plus** the two `.py` scripts. Also `+ Add input` the **RDD-2022** dataset (needed to build `rapi5` + `china_v5`).

If your zip already contains a `rapiernet_hr2/` dir at top level, the restore cell adapts automatically.

In [ ]:
# 1) dependencies -----------------------------------------------
import subprocess, sys
def _have(pkg):
    return subprocess.run([sys.executable, '-c', f'import {pkg}'],
                          capture_output=True).returncode == 0
if not _have('ultralytics') or not _have('cv2'):
    # 8.4.146 = the exact version that produced the final 0.862 run on Kaggle.
    # (8.3.283 is NOT on PyPI; it only exists as the local pre-release.
    #  Use the proven Kaggle version instead.)
    tried = []
    for pin in ('ultralytics==8.4.146', 'ultralytics'):
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                            '--no-cache-dir', pin], stderr=subprocess.STDOUT)
        tried.append((pin, r.returncode))
        if r.returncode == 0 and _have('ultralytics'):
            break
    print('install attempts:', tried)
print('deps ok')


In [ ]:
# 2) restore scripts + final seed into /kaggle/working ----------------
import glob, os, shutil, gc
from pathlib import Path
W = Path('/kaggle/working'); W.mkdir(exist_ok=True)

# --- checkpoint dataset (exact Kaggle paths) ---
CHK = Path('/kaggle/input/datasets/amlmoawad/rdd2022-stage2-checkpoints')
assert CHK.is_dir(), f'checkpoint dataset not found: {CHK}'
(W / 'rapiernet_hr2' / 'weights').mkdir(parents=True, exist_ok=True)
for f in ('best.pt', 'last.pt', 'args.yaml', 'results.csv'):
    src, dst = CHK / f, None
    if f.endswith('.pt'):
        dst = W / 'rapiernet_hr2' / 'weights' / f
    else:
        dst = W / 'rapiernet_hr2' / f
    if src.exists():
        shutil.copy2(str(src), str(dst))
        print('restored', f)
    else:
        print('MISSING in checkpoint dataset:', f)
SEED = str(W / 'rapiernet_hr2' / 'weights' / 'best.pt')

# --- scripts (search all inputs, fall back to checkpoint dataset) ---
def _find_script(name):
    for f in glob.glob('/kaggle/input/**/' + name, recursive=True):
        return f
    return None
for sp in ('rapiernet_rdd2022_final.py', 'make_paper_figures.py'):
    src = _find_script(sp)
    if src:
        shutil.copy2(src, str(W / sp))
        print('script ->', sp)
    else:
        print('!! script not found in any input:', sp)
        print('   -> add it to the "datasets/amlmoawad/rdd2022-stage2-checkpoints" dataset (as a regular file)')

print('seed:', SEED, '| exists:', os.path.isfile(SEED))
print('scripts:', (W/'rapiernet_rdd2022_final.py').exists(), (W/'make_paper_figures.py').exists())
assert os.path.isfile(SEED), 'final seed best.pt not found'
assert (W / 'rapiernet_rdd2022_final.py').exists(), 'missing rapiernet_rdd2022_final.py'


In [ ]:
# 3) environment ------------------------------------------------
import torch, ultralytics, gc
print('cuda:', torch.cuda.is_available(), '| cuda_dev:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')
print('torch', torch.__version__, '| ultralytics', ultralytics.__version__)
gc.collect()


In [ ]:
# 4) load pipeline (registers custom modules, exposes build_abl / eval helpers) --
import runpy
W = Path('/kaggle/working')
ns = runpy.run_path(str(W / 'rapiernet_rdd2022_final.py'), run_name='not_main')
ns['CUR_NC'] = 5
ns['DATA_NAME'] = 'rapi5'
print('protocol constants:', ns.get('DATA_NAME'), ns.get('CUR_NC'))


In [ ]:
# 5) build rapi5 (train) + china_v5 (protocol test) --------------//-
import glob, os
W = Path('/kaggle/working')
RDD = Path('/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT')
if (W / 'RDD_SPLIT').is_dir():
    ns['DATASET'] = str(W / 'RDD_SPLIT')
elif RDD.is_dir():
    ns['DATASET'] = str(RDD)
else:
    raise FileNotFoundError('RDD_SPLIT not found (expected /kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT)')
print('dataset root ->', ns['DATASET'])
_m4, _rapi5 = ns['prepare_datasets']()
ns['build_china_v5']()
print('yaml ok:', (W/'rapi5.yaml').exists(), (W/'china_v5.yaml').exists())


In [ ]:
# 6) ablation configuration --------------------------------------
EPOCHS = 6        # fine-tune epochs per variant (resume-safe)
IMGSZ  = 672      # protocol resolution
BATCH  = 8        # matches final run on T4 (nbs=64)
CONF   = 0.05     # final operating point
DEV    = 0
SEED   = str(W / 'rapiernet_hr2' / 'weights' / 'best.pt')
NAMES = ['alligator','transverse','longitudinal','corrupt','pothole']

CONFIGS = [
    ('Full RapierNet',   dict(use_dysample=True,  use_coord=True,  use_sarf=True)),
    ('w/o SARF',         dict(use_dysample=True,  use_coord=True,  use_sarf=False)),
    ('w/o CoordGate',    dict(use_dysample=True,  use_coord=False, use_sarf=True)),
    ('w/o DySample',     dict(use_dysample=False, use_coord=True,  use_sarf=True)),
    ('w/o SARF+Coord',   dict(use_dysample=True,  use_coord=False, use_sarf=False)),
    ('Plain yolo11-m',   dict(use_dysample=False, use_coord=False, use_sarf=False)),
]
print('configs:')
[print('  ', n, f) for n, f in CONFIGS]


In [ ]:
# 7) TTA protocol eval (same as final reporting) --------------------
def _metric_block(met):
    f1 = 2 * met.box.mp * met.box.mr / (met.box.mp + met.box.mr + 1e-9)
    per = {n: round(float(ap), 3) for n, ap in zip(NAMES, met.box.ap50)}
    return dict(mAP50=round(float(met.box.map50), 3),
                mAP50_95=round(float(met.box.map), 3),
                F1=round(float(f1), 3), per_class=per)

def ev(ckpt, conf=CONF, imgsz=IMGSZ, tta=True, device=DEV):
    from ultralytics import YOLO
    _, yf = ns['build_china_v5']()
    m = YOLO(ckpt)
    met = m.val(data=yf, conf=conf, iou=0.5, imgsz=imgsz, device=device,
                batch=8, verbose=False, augment=tta)
    return _metric_block(met)

def ev_obj(y, conf=CONF, imgsz=IMGSZ, tta=True, device=DEV):
    """Evaluate an in-memory YOLO object (no save/load round-trip)."""
    _, yf = ns['build_china_v5']()
    met = y.val(data=yf, conf=conf, iou=0.5, imgsz=imgsz, device=device,
                batch=8, verbose=False, augment=tta)
    return _metric_block(met)


In [ ]:
# 8) run ablation table (culprit-proof: in-place surgery on the seed) --
import os, gc, glob, shutil, torch, time, csv
import torch.nn as nn
from copy import deepcopy
from ultralytics import YOLO

# wipe any stale variant dirs from earlier broken runs (no resume of garbage)
for d in glob.glob('/kaggle/working/ablchina_*'):
    shutil.rmtree(d, ignore_errors=True)

def build_variant(seed, flags):
    """Load the seed checkpoint (self.ckpt set -> framework fine-tune path),
    swap SARF/CoordGate/DySample OFF in place, AND repoint y.model.yaml to
    the variant config. YOLO.train() then rebuilds (engine/model.py:852) the
    variant model and re-transfers the seed weights from the MUTATED module."""
    use_dysample = flags.get('use_dysample', True)
    use_coord    = flags.get('use_coord',    True)
    use_sarf     = flags.get('use_sarf',     True)
    y = YOLO(seed)                    # self.ckpt is set -> line-852 rebuild path
    cfg = deepcopy(y.model.yaml)      # nc=5 kept (fixes the old nc=4 head drop)
    hd = cfg['head'] if isinstance(cfg['head'], list) else cfg['head']
    for layer in hd:
        if isinstance(layer, list) and len(layer) >= 2:
            t = layer[2]
            if t == 'DySample' and not use_dysample:
                layer[2] = 'nn.Upsample'; layer[3] = [None, 2, 'nearest']
            elif t == 'CoordGate' and not use_coord:
                layer[2] = 'nn.Identity'
            elif t == 'SARF' and not use_sarf:
                layer[2] = 'nn.Identity'
    # in-place module swap (with ultralytics attrs so ev_obj can forward)
    for i, blk in enumerate(y.model.model):
        cn = blk.__class__.__name__
        newmod = None
        if cn == 'DySample' and not use_dysample:
            newmod = nn.Upsample(scale_factor=2, mode='nearest')
        elif cn == 'CoordGate' and not use_coord:
            newmod = nn.Identity()
        elif cn == 'SARF' and not use_sarf:
            newmod = nn.Identity()
        if newmod is not None:
            newmod.f = getattr(blk, 'f', -1); newmod.i = getattr(blk, 'i', i); newmod.np = 0
            y.model.model[i] = newmod
    y.model.yaml = cfg                # trainer rebuild now uses the VARIANT config
    return y

def train_variant(name, flags, seed=SEED, force=False):
    rn = 'ablchina_' + name.replace(' ', '_').replace('/', '_')
    run_dir = str(W / rn)
    last_pt = run_dir + '/weights/last.pt'; best_pt = run_dir + '/weights/best.pt'
    if os.path.isfile(best_pt) and not force:
        print('[reuse ]', rn)  # already trained this variant -> just re-eval
        return best_pt
    if os.path.isfile(last_pt):
        ck = torch.load(last_pt, map_location='cpu', weights_only=False)
        rs = (ck.get('epoch', -1) >= 0) and (ck.get('optimizer') is not None)
        del ck; gc.collect()
        if rs:
            print('[resume]', rn)
            YOLO(last_pt).train(resume=True, data=str(W / 'rapi5.yaml'),
                                imgsz=IMGSZ, batch=BATCH, device=DEV, workers=4)
            return best_pt
    v = build_variant(seed, flags)
    # sanity: the ablated model must START near the Full score, not ~0
    r0 = ev_obj(v)
    print('>>> pre-train eval %s: mAP50=%.3f mAP50-95=%.3f (full=0.862)'
          % (name, r0['mAP50'], r0['mAP50_95']))
    v.train(data=str(W / 'rapi5.yaml'), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
            device=DEV, workers=4, seed=42, deterministic=False, cache=False,
            patience=40, project=str(W), name=rn, exist_ok=True, save_period=10,
            lr0=1e-3, lrf=0.2, optimizer='SGD', momentum=0.937, weight_decay=0.0005,
            warmup_epochs=1.0, cos_lr=True, close_mosaic=5, amp=True, val=True,
            plots=False, mosaic=1.0, mixup=0.15, copy_paste=0.2, degrees=10.0,
            perspective=0.0001, flipud=0.1, fliplr=0.5, hsv_h=0.015, hsv_s=0.7,
            hsv_v=0.4, scale=0.5, translate=0.1, erasing=0.3, cls=0.8,
            box=7.5, dfl=1.5)
    return best_pt

rows = []
for name, flags in CONFIGS:
    t0 = time.time()
    ckpt = train_variant(name, flags)
    if not os.path.isfile(ckpt):
        print('!!! missing best.pt for', name); continue
    r = ev(ckpt)
    print('>>> %-18s mAP50=%.3f mAP50-95=%.3f F1=%.3f  (%d min)'
          % (name, r['mAP50'], r['mAP50_95'], r['F1'], (time.time()-t0)//60))
    rows.append(dict(config=name, epochs=EPOCHS, mAP50=r['mAP50'],
                     mAP50_95=r['mAP50_95'], F1=r['F1'], **r['per_class']))
    with open(str(W / 'ablation_table_china.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())
            if rows else ['config','epochs','mAP50','mAP50_95','F1'] + NAMES)
        w.writeheader(); w.writerows(rows)
print('\nsaved ablation_table_china.csv')


In [ ]:
# 9) table + bar figure --------------------------------------------
import csv, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
W = Path('/kaggle/working')
OUT = str(W / 'paper_figs'); os.makedirs(OUT, exist_ok=True)

with open(str(W / 'ablation_table_china.csv')) as f:
    rows = list(csv.DictReader(f))
print('%-18s %7s %9s %6s' % ('config', 'mAP50', 'mAP50-95', 'F1'))
print('-' * 46)
for r in rows:
    print('%-18s %7.3f %9.3f %6.3f' % (r['config'], float(r['mAP50']),
          float(r['mAP50_95']), float(r['F1'])))

rs = sorted(rows, key=lambda r: -float(r['mAP50']))
names = [r['config'] for r in rs]
vals = [float(r['mAP50']) for r in rs]
fig, ax = plt.subplots(figsize=(7.6, 4.2))
bars = ax.barh(names, vals, color='#4C72B0')
for b, v in zip(bars, vals):
    ax.text(v + 0.003, b.get_y() + b.get_height()/2, '%.3f' % v,
            va='center', fontsize=9)
ax.set_xlabel('mAP50 (China protocol, TTA @672, conf 0.05)')
ax.set_xlim(0, max(vals) + 0.05)
ax.set_title('RapierNet component ablation (seeded from final best.pt, %s epochs)' % rows[0]['epochs'])
fig.tight_layout()
for ext in ('pdf', 'png'):
    fig.savefig(f'{OUT}/fig_ablation_china.{ext}', dpi=300, bbox_inches='tight')
plt.close(fig)
print('saved fig_ablation_china.{pdf,png}')


## 10) Download

Zip `ablation_table_china.csv`, the variant run dirs (`ablchina_*`), and the figure before ending the session:
```python
import zipfile, os, glob
src = '/kaggle/working'
items = ['ablation_table_china.csv', 'paper_figs'] + sorted(glob.glob(src + '/ablchina_*'))
zf = zipfile.ZipFile(src + '/ablation_output.zip', 'w', zipfile.ZIP_DEFLATED)
for name in items:
    if not os.path.exists(name) or not name.startswith(src):
        continue
    for root, _, files in os.walk(name):
        if any(x in root for x in ('Dockerfile', '.ipynb_checkpoints')):
            continue
        for f in files:
            if f.endswith(('.pt', '.csv', '.yaml', '.pdf', '.png')) and not f.endswith('yolo11m.pt'):
                fp = os.path.join(root, f)
                zf.write(fp, os.path.relpath(fp, src))
zf.close()
print('zipped -> /kaggle/working/ablation_output.zip')
```
